# DiLM Baselines — SST-2 / BERT base uncased

Сравнение методов дистилляции датасета на одной downstream-задаче (классификация sentiment'а SST-2 → BERT).

Идея dataset distillation: вместо тренировки на полном датасете (67k примеров) хотим получить **крошечный** синтетический/выбранный набор (`DPC` примеров на класс), на котором BERT с нуля обучается до сопоставимой точности.

**Пайплайн каждого метода** (все шаги явно прописаны в клетках ниже):
1. **distill** — получить `N_DATASET` независимых дистиллированных датасетов по `DPC` примеров на класс
2. **evaluate** — для каждого датасета `n_eval_per_dataset` раз обучить BERT с нуля и замерить accuracy на валидации
3. **summarize** — mean ± std по всем прогонам → `results/{task}/{method}/{run}/summary.json`

Таблица сравнения внизу собирает все `summary.json` под `results/{task}/`.

**Требует CUDA** — `Evaluator` и тренеры DiLM зовут `.cuda()`.

## 1. Setup

`dilm_wrapper.py` содержит только билдеры компонент DiLM + мелкие хелперы. Логика пайплайна — в самом ноутбуке, чтобы было что менять руками.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import mlflow
from transformers import set_seed

from dilm_wrapper import (
    build_learner, build_generator, build_data_module, build_evaluator,
    build_coreset_module, prepare_repset_teachers,
    build_trainer_lm, build_trainer_dc, generate_dilm_datasets,
    summarize, save_summary, collect_summaries,
    RESULTS_ROOT,
)

## 2. Гиперпараметры

Все ручки в одном месте. Поменять `LEARNER_MODEL` / `TASK` чтобы переключить модель или датасет (см. *Extending* в конце).

- `DPC` — distilled-per-class. Бюджет дистиллированного датасета на класс. Для SST-2 (2 класса) при `DPC=20` итоговый датасет = 40 примеров.
- `N_DATASET` — сколько **независимых** дистиллированных датасетов получает каждый метод. Усредняет шум самой дистилляции (разные seeds в random / sampling в GPT-2).
- `EVAL_KW.n_eval_per_dataset` — сколько раз переобучаем BERT на одном и том же дистиллированном датасете. Усредняет шум инициализации классификационной головы. Итого: `N_DATASET × n_eval_per_dataset` обучений BERT, и mean/std считается по всем.

In [ ]:
TASK = "sst2"
LEARNER_MODEL = "bert-base-uncased"

DPC = 20         # дистиллированных примеров на класс
N_DATASET = 2    # сколько независимых дистиллированных наборов получаем
SEED = 42

EVAL_KW = dict(
    train_step=200,        # шагов файнтюна BERT на одном eval-прогоне
    batch_size=64,
    lr=1e-4,
    n_eval_per_dataset=2,  # переобучаем BERT столько раз на каждом дистиллированном наборе
)

RUN_NAME = f"dpc{DPC}_seed{SEED}"
set_seed(SEED)

mlflow.set_tracking_uri(f"file:{RESULTS_ROOT}/mlruns")
mlflow.set_experiment(f"baselines.{TASK}")

## 3. Датасет

`DataModule` качает GLUE/SST-2, переименовывает `label`→`labels`, токенизирует **двумя** токенизаторами одновременно (генератор GPT-2 и learner BERT) и кладёт результат на диск. Двойная токенизация нужна, потому что один и тот же `DataModule` обслуживает все методы — даже coreset-методы получают батчи в формате `{generator: ..., learner: ...}`, хотя ветка `generator` для них не используется.

In [ ]:
learner = build_learner(LEARNER_MODEL, TASK)
generator = build_generator(TASK)  # gpt2, веса "из коробки"
data_module = build_data_module(TASK, learner, generator)

print(f"train:  {len(data_module.datasets['train']):>6}")
print(f"valid:  {len(data_module.datasets['validation']):>6}")
print(f"labels: {data_module.num_labels}")
data_module.datasets['train'][0]

## 4. Evaluator

Один объект `Evaluator` инкапсулирует процедуру "переобучить BERT с нуля на дистиллированном датасете → замерить accuracy на validation SST-2". Используется и всеми методами ниже, и **внутри** DiLM-тренеров для mid-training валидации генератора.

In [ ]:
evaluator = build_evaluator(TASK, **EVAL_KW)
METRIC_KEY = evaluator.metric_key  # "accuracy" для sst2
METRIC_KEY

## 5. Coreset-бейзлайны

**Идея:** вместо синтеза текстов выбрать `DPC` *реальных* примеров на класс по умному критерию. Никакого обучения генератора — только эмбеддинги BERT (для embedding-based методов) и арифметика над ними.

Все три селектора в DiLM работают **по классам отдельно**: фильтруем train по `labels==c`, выбираем `DPC` примеров, конкатенируем.

Каждый блок ниже: построить селектор → получить `N_DATASET` дистиллированных датасетов → `evaluator.evaluate` → сохранить summary.

### 5.1 Random

**Что делает.** Просто берёт случайные `DPC` примеров на класс. Контрольный бейзлайн — любой осмысленный метод должен его побить.

**Как работает.** `dataset.shuffle(seed=i).select(range(dpc))` для каждого из `N_DATASET` разных seeds. Никаких эмбеддингов, никаких моделей — просто i.i.d. сэмпл из категориального равномерного распределения.

**Артефакты.** `data/sst2/coresets/random/coreset_dcp_20_version_{i}.json` (кэшируется — при повторе берётся с диска).

In [ ]:
method = "random"
run_dir = RESULTS_ROOT / TASK / method / RUN_NAME

selector = build_coreset_module(TASK, method, data_module)
distilled = selector.generate_dataset(dpc=DPC, n=N_DATASET)
print(f"{len(distilled)} датасетов, в первом {len(distilled[0])} строк")
distilled[0][:3]

In [ ]:
with mlflow.start_run(run_name=f"{method}.{RUN_NAME}"):
    mlflow.log_params({"method": method, "dpc": DPC, "n_dataset": N_DATASET, "seed": SEED})
    results = evaluator.evaluate(
        dataset_list=distilled,
        learner=learner,
        data_module=data_module,
        save_result_dir=str(run_dir / "metrics"),
        verbose=True,
    )

summary = summarize(results, METRIC_KEY, name=RUN_NAME, method=method, task=TASK,
                    learner=LEARNER_MODEL, dpc=DPC, n_dataset=N_DATASET)
save_summary(summary, run_dir)
summary

### 5.2 k-centers

**Что делает.** Выбирает `DPC` примеров на класс так, чтобы они **максимально покрывали** пространство эмбеддингов (geometric coverage). Цель — чтобы у BERT, обучающегося на этом наборе, не было "слепых пятен" в распределении.

**Как работает.**
1. Прогоняем все примеры класса через предобученный BERT, берём `[CLS]`-эмбеддинг → точки в пространстве размерности 768.
2. Жадный farthest-point sampling: первый центр — случайный, каждый следующий — точка, у которой расстояние до ближайшего уже выбранного центра максимальное. Это аппроксимация задачи k-центров (NP-hard).
3. Повторяем `N_DATASET` раз с разными seeds для starting point.

**Артефакты.** `data/sst2/coresets/k_centers/bert-base-uncased/coreset_dcp_20_version_{i}.json`. Эмбеддинги кэшируются.

In [ ]:
method = "k_centers"
run_dir = RESULTS_ROOT / TASK / method / RUN_NAME

selector = build_coreset_module(TASK, method, data_module)
distilled = selector.generate_dataset(dpc=DPC, n=N_DATASET)

with mlflow.start_run(run_name=f"{method}.{RUN_NAME}"):
    mlflow.log_params({"method": method, "dpc": DPC, "n_dataset": N_DATASET, "seed": SEED})
    results = evaluator.evaluate(
        dataset_list=distilled, learner=learner, data_module=data_module,
        save_result_dir=str(run_dir / "metrics"), verbose=True,
    )

summary = summarize(results, METRIC_KEY, name=RUN_NAME, method=method, task=TASK,
                    learner=LEARNER_MODEL, dpc=DPC, n_dataset=N_DATASET)
save_summary(summary, run_dir)
summary

### 5.3 Herding

**Что делает.** Выбирает `DPC` примеров на класс так, чтобы их **среднее эмбеддинга совпадало** со средним эмбеддингом всего класса. Цель — сохранить "центр масс" распределения класса.

**Как работает.**
1. Те же `[CLS]`-эмбеддинги BERT.
2. Считаем mean-вектор `μ` по всему классу.
3. Жадно по одному добавляем примеры так, чтобы среднее уже выбранных приближалось к `μ`: на каждом шаге берём пример, минимизирующий `‖μ − mean(selected ∪ {x})‖`. Детерминированно (поэтому herding не выигрывает от разных seeds, все `N_DATASET` версии одинаковые — но мы всё равно их строим для единообразия pipeline'а).

**Контраст с k-centers.** k-centers покрывает периферию (включает outliers), herding концентрируется на типичных примерах вокруг центра.

In [ ]:
method = "herding"
run_dir = RESULTS_ROOT / TASK / method / RUN_NAME

selector = build_coreset_module(TASK, method, data_module)
distilled = selector.generate_dataset(dpc=DPC, n=N_DATASET)

with mlflow.start_run(run_name=f"{method}.{RUN_NAME}"):
    mlflow.log_params({"method": method, "dpc": DPC, "n_dataset": N_DATASET, "seed": SEED})
    results = evaluator.evaluate(
        dataset_list=distilled, learner=learner, data_module=data_module,
        save_result_dir=str(run_dir / "metrics"), verbose=True,
    )

summary = summarize(results, METRIC_KEY, name=RUN_NAME, method=method, task=TASK,
                    learner=LEARNER_MODEL, dpc=DPC, n_dataset=N_DATASET)
save_summary(summary, run_dir)
summary

## 6. DiLM — обучить генератор и сгенерировать дистиллированные данные

**Принципиальное отличие от coreset.** Coreset *выбирает* реальные примеры. DiLM *синтезирует новые тексты* через языковую модель, которая обучается так, чтобы её сэмплы были "информативны" для downstream learner'а.

**Зачем это нужно.** Coreset ограничен подмножеством реального датасета — у real примера может быть лишняя информация, или его нельзя "сжать" сильнее, чем одна точка датасета. Generator может выдавать тексты, которые "плотнее" по информации для классификатора. Кроме того, синтетические данные text-level (а не embedding-level, как в предыдущих работах) переносимы между разными классификаторами.

**Три стадии:**
- **6.3 LM pretrain** — файнтюним GPT-2 на real-предложениях SST-2 как причинную ЯМ, но с label-conditioning: к каждому предложению префиксом добавляется `<bos_i>` где `i` — метка класса. После этой стадии генератор умеет производить класс-условные сэмплы, но без знания о том, что важно для downstream BERT.
- **6.4 GM fine-tune (gradient matching).** Главный шаг DiLM. На каждой итерации:
  1. сэмплим батч real-примеров класса `c` (из k-centers "teacher" coreset, см. ниже)
  2. считаем градиент BERT по cross-entropy на этом батче — назовём его `g_real`
  3. сэмплим из генератора батч синтетических примеров класса `c`
  4. считаем градиент BERT на синтетике взвешенно (вес каждого синт-примера = softmax от его log-prob под генератором, нормированный температурой) — `g_syn`
  5. **минимизируем** `1 − cos(g_real, g_syn)` по параметрам **генератора**. Градиент течёт через веса синт-примеров (т.е. через лог-вероятность того, что генератор такие тексты сэмплит) — это и есть способ обучить ЯМ через дискретный текст без REINFORCE.
- **6.5 Generate + prune.** Когда генератор обучен — сэмплим `DPC × OVER_SAMPLE` (по умолчанию × 100) на класс, потом k-centers'ом обрезаем до `DPC` для разнообразия.

**Заметка про стоимость.** Paper-бюджеты: LM 80k шагов, DC 20k шагов — это часы на A100. Дефолты ниже игрушечные (200/200) — pipeline отработает быстро, но **accuracy будет плохая**. Подними `LM_STEPS` / `DC_STEPS` до paper-значений для реальных чисел.

### 6.1 Бюджет обучения

In [ ]:
# Ограничения TrainerDC: DC_STEPS % INNER_LOOP == 0,
# DC_VAL_INTERVAL % INNER_LOOP == 0, (DC_STEPS//INNER_LOOP) % (DC_VAL_INTERVAL//INNER_LOOP) == 0.
LM_STEPS = 80000            # paper: 80000
DC_STEPS = 20000            # paper: 20000
INNER_LOOP = 10           # сколько шагов GM-апдейта генератора в одном outer-loop
MODEL_STEP_PER_INNER = 20 # сколько шагов обновления learner-а между шагами генератора
GENERATE_INTERVAL = 10    # как часто (в outer-loops) пересэмплируем синт-данные генератором
GM_SYN_DPC = 64           # размер синт-батча на класс для GM-loss
GM_REAL_DPC = 200         # размер real-батча на класс для GM-loss
REPSET_DPC = 200          # размер одного teacher-coreset на класс
N_REPSET = 10             # сколько k-centers teacher-coresets используется как gradient-teachers
OVER_SAMPLE = 100.0       # сэмплим DPC*OVER_SAMPLE на класс, потом обрезаем k-centers'ом до DPC

dilm_dir = RESULTS_ROOT / TASK / "dilm" / RUN_NAME

### 6.2 Teacher-coreset + вспомогательный k-centers селектор

Один и тот же k-centers селектор закрывает две роли:
1. **Teacher-coresets для GM-loss.** Генерим `N_REPSET` независимых coreset-датасетов по `REPSET_DPC` примеров на класс. На каждом шаге GM-loss `g_real` считается на одном из этих teacher'ов (циклически). Зачем не просто рандом из train: фиксированный осмысленный референс стабилизирует gradient matching.
2. **Pruning сгенерированных текстов** (используется и в 6.5, и **внутри** тренера для mid-training валидации): из `DPC*OVER_SAMPLE` синт-примеров выбираем `DPC` farthest-point'ов по их BERT-эмбеддингам.

In [ ]:
kc_module = build_coreset_module(TASK, "k_centers", data_module)
repset_teachers = prepare_repset_teachers(kc_module, data_module, REPSET_DPC, N_REPSET)
print(f"{len(repset_teachers)} teacher-coresets, в каждом {len(repset_teachers[0])} примеров")

### 6.3 LM pretrain

Файнтюн GPT-2 на реальных предложениях SST-2 как причинной языковой модели с label-conditioning. К embeddings GPT-2 добавляются специальные токены `<bos_0>`, `<bos_1>` (по числу классов) и `<sep>` — генератор учится продолжать `<bos_i>` так, чтобы получалось предложение класса `i`.

`TrainerLM.fit` мутирует генератор inplace и в конце грузит обратно best-checkpoint. Сохраняет в `{dilm_dir}/lm/generator/{best,last}-ckpt` + tokenizer.

In [ ]:
lm_dir = dilm_dir / "lm"
trainer_lm = build_trainer_lm(
    lm_dir,
    total_train_step=LM_STEPS,
    val_interval=max(LM_STEPS, 1),  # отключаем mid-train валидацию на игрушечных бюджетах
    val_skip_step=LM_STEPS + 1,
)

with mlflow.start_run(run_name=f"dilm_lm.{RUN_NAME}"):
    mlflow.log_params({"stage": "lm", "steps": LM_STEPS, "seed": SEED})
    trainer_lm.fit(
        generator=generator,
        learner=learner,
        data_module=data_module,
        evaluator=evaluator,
        repset_teachers=None,
        coreset_module=kc_module,
    )

### 6.4 GM fine-tune (gradient matching)

Главная стадия DiLM. `TrainerDC.fit` продолжает обучение уже LM-предобученного генератора. На каждом outer-loop:
- (раз в `GENERATE_INTERVAL` outer-loops) пересэмплируется пул синт-данных из текущего генератора
- `INNER_LOOP` шагов обновления генератора по GM-loss
- между ними `MODEL_STEP_PER_INNER` шагов обновления learner'а на real-данных (чтобы матчинг был не только относительно случайного init BERT, но и относительно "работающих" весов)

Сохраняет в `{dilm_dir}/dc/generator/{best,last}-ckpt`.

In [ ]:
dc_dir = dilm_dir / "dc"
trainer_dc = build_trainer_dc(
    dc_dir,
    total_train_step=DC_STEPS,
    inner_loop=INNER_LOOP,
    model_step_per_inner_step=MODEL_STEP_PER_INNER,
    gm_syn_dpc=GM_SYN_DPC,
    gm_real_dpc=GM_REAL_DPC,
    repset_dpc=REPSET_DPC,
    n_repset=N_REPSET,
    generate_dataset_interval=GENERATE_INTERVAL,
    val_interval=DC_STEPS,
    val_skip_step=DC_STEPS + 1,
    log_interval=DC_STEPS,
    dpc=DPC,
    n_dataset=N_DATASET,
    over_sample_ratio=OVER_SAMPLE,
)

with mlflow.start_run(run_name=f"dilm_dc.{RUN_NAME}"):
    mlflow.log_params({"stage": "dc", "steps": DC_STEPS, "inner_loop": INNER_LOOP, "seed": SEED})
    trainer_dc.fit(
        generator=generator,
        learner=learner,
        data_module=data_module,
        evaluator=evaluator,
        repset_teachers=repset_teachers,
        coreset_module=kc_module,
    )

### 6.5 Генерация дистиллированных датасетов

Обученный генератор уже может выдавать класс-условные тексты. Сэмплим `DPC × OVER_SAMPLE = 2000` на класс, потом k-centers'ом обрезаем до `DPC = 20` для разнообразия. Получаем `N_DATASET` независимых дистиллированных датасетов.

In [ ]:
distilled = generate_dilm_datasets(
    generator=generator,
    coreset_module=kc_module,
    dpc=DPC,
    n_dataset=N_DATASET,
    over_sample_ratio=OVER_SAMPLE,
    save_dir=dilm_dir / "distilled_datasets",
)
print(f"сгенерировано {len(distilled)} датасетов, по {len(distilled[0])} строк")
distilled[0][:3]

### 6.6 Evaluate + summary

Тот же `Evaluator`, что и для coreset-методов — единый протокол замера.

In [ ]:
method = "dilm"
run_dir = dilm_dir  # alias

with mlflow.start_run(run_name=f"{method}.{RUN_NAME}"):
    mlflow.log_params({"method": method, "dpc": DPC, "n_dataset": N_DATASET,
                       "lm_steps": LM_STEPS, "dc_steps": DC_STEPS, "seed": SEED})
    results = evaluator.evaluate(
        dataset_list=distilled, learner=learner, data_module=data_module,
        save_result_dir=str(run_dir / "metrics"), verbose=True,
    )

summary = summarize(results, METRIC_KEY, name=RUN_NAME, method=method, task=TASK,
                    learner=LEARNER_MODEL, dpc=DPC, n_dataset=N_DATASET,
                    lm_steps=LM_STEPS, dc_steps=DC_STEPS)
save_summary(summary, run_dir)
summary

## 7. Таблица сравнения

Собирает `results/{task}/**/summary.json`. Можно копить между запусками — таблица растёт.

In [ ]:
df = collect_summaries(TASK)
cols = ["method", "name", "learner", "dpc", "n_dataset", f"{METRIC_KEY}_mean", f"{METRIC_KEY}_std"]
df[cols].sort_values(f"{METRIC_KEY}_mean", ascending=False).reset_index(drop=True)